In [3]:
import time
import numpy as np
import pandas as pd
from numba import njit

# 原始版本
def original_version(s):
    v = s.values
    n = len(v)
    last_greater = np.full(n, -1)
    stack = []
    counts = np.zeros(n, dtype=int)

    for i in range(n):
        while stack and v[stack[-1]] < v[i]:
            stack.pop()
        if stack:
            last_greater[i] = stack[-1]
        stack.append(i)
        if i == 0:
            counts[i] = 0
        else:
            counts[i] = (i - (last_greater[i] + 1)) if last_greater[i] != -1 else i
    
    return pd.Series(counts, index=s.index)

# Numba 版本
@njit
def numba_core(values):
    n = len(values)
    last_greater = np.full(n, -1)
    stack = []
    counts = np.zeros(n, dtype=np.int64)

    for i in range(n):
        while stack and values[stack[-1]] < values[i]:
            stack.pop()
        if stack:
            last_greater[i] = stack[-1]
        stack.append(i)
        if i == 0:
            counts[i] = 0
        else:
            counts[i] = (i - (last_greater[i] + 1)) if last_greater[i] != -1 else i
    
    return counts

def numba_version(s):
    counts = numba_core(s.values)
    return pd.Series(counts, index=s.index)

# 性能测试
def test_performance(n_samples=1000000):
    # 生成测试数据
    data = np.random.randint(1, 100, n_samples)
    s = pd.Series(data)
    
    # 预热 Numba
    _ = numba_version(s[:100])
    
    # 测试原始版本
    start = time.time()
    original_version(s)
    original_time = time.time() - start
    
    # 测试 Numba 版本
    start = time.time()
    numba_version(s)
    numba_time = time.time() - start
    
    print(f"数据量: {n_samples}")
    print(f"原始版本耗时: {original_time:.6f}秒")
    print(f"Numba版本耗时: {numba_time:.6f}秒")
    print(f"性能提升: {original_time/numba_time:.2f}倍")

# 测试不同数据规模
for n in [10, 100, 1000, 10000, 100000, 1000000]:
    print(f"\n{'='*20}")
    test_performance(n)


数据量: 10
原始版本耗时: 0.000145秒
Numba版本耗时: 0.000061秒
性能提升: 2.38倍

数据量: 100
原始版本耗时: 0.000229秒
Numba版本耗时: 0.000056秒
性能提升: 4.09倍

数据量: 1000
原始版本耗时: 0.001610秒
Numba版本耗时: 0.000215秒
性能提升: 7.49倍

数据量: 10000
原始版本耗时: 0.016609秒
Numba版本耗时: 0.000398秒
性能提升: 41.71倍

数据量: 100000
原始版本耗时: 0.140724秒
Numba版本耗时: 0.002225秒
性能提升: 63.25倍

数据量: 1000000
原始版本耗时: 1.391116秒
Numba版本耗时: 0.022205秒
性能提升: 62.65倍
